<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/2026-2027/5_2_Nonlinear_Nonparametric_Models_Bagging_and_Random_Forest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Part V. Nonlinear Nonparametric Models.
## 1. Decision Trees (CART)

## 2. Bagging and Random Forest

## 3. Gradient Boosting (XGBoost, LightGBM, CatBoost)


## 5.2. Бэггинг (Bootstrap Aggregating)

Одиночные решающие деревья страдают от высокой дисперсии: небольшие изменения в обучающей выборке способны радикально перестроить структуру дерева. Бэггинг (Bootstrap Aggregating), предложенный Брейманом (Breiman, 1996), уменьшает дисперсию предсказаний за счёт усреднения большого числа деревьев, обученных на случайно порождённых версиях исходных данных. Этот раздел посвящён математическим основаниям бэггинга, его свойствам и практической реализации.

### 1. Идея бэггинга

Пусть имеется обучающая выборка $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$. Бэггинг строит ансамбль из $B$ базовых моделей (как правило, решающих деревьев), каждая из которых обучается на **бутстреп‑выборке** $\mathcal{D}_b^*$, полученной из $\mathcal{D}$ случайным выбором $n$ объектов **с возвращением**. Таким образом, в каждой бутстреп‑выборке некоторые объекты исходной выборки могут присутствовать несколько раз, а некоторые — отсутствовать. Вероятность для конкретного объекта не попасть в $\mathcal{D}_b^*$ равна

$$
\left(1 - \frac{1}{n}\right)^n \xrightarrow{n\to\infty} e^{-1} \approx 0{,}368.
$$

Следовательно, каждая бутстреп‑выборка содержит в среднем около $63{,}2\%$ уникальных объектов исходной выборки.

Ансамблевое предсказание для регрессии строится как среднее арифметическое предсказаний отдельных моделей:

$$
\hat{f}_{\text{bag}}(\mathbf{x}) = \frac{1}{B} \sum_{b=1}^{B} \hat{f}_b(\mathbf{x}),
$$

где $\hat{f}_b$ — модель, обученная на $\mathcal{D}_b^*$. В задаче классификации агрегирование обычно проводят голосованием: большинством (hard voting) или усреднением предсказанных вероятностей классов (soft voting).

### 2. Математическое обоснование снижения дисперсии

Эффективность бэггинга основана на уменьшении дисперсии предсказаний за счёт усреднения. Рассмотрим идеализированную ситуацию: имеется $B$ случайных величин $\hat{f}_1(\mathbf{x}),\dots,\hat{f}_B(\mathbf{x})$, представляющих предсказания моделей ансамбля для фиксированной точки $\mathbf{x}$. Предположим, что все они имеют одинаковое математическое ожидание $\mathbb{E}[\hat{f}_b] = \mu$, одинаковую дисперсию $\mathrm{Var}(\hat{f}_b) = \sigma^2$ и попарную корреляцию $\rho = \mathrm{Corr}(\hat{f}_i, \hat{f}_j)$ для $i \neq j$. Тогда для среднего $\bar{f} = \frac{1}{B}\sum_{b=1}^B \hat{f}_b$ имеем:

$$
\begin{aligned}
\mathrm{Var}(\bar{f})
&= \frac{1}{B^2} \left( \sum_{i=1}^{B} \mathrm{Var}(\hat{f}_i) + \sum_{i \neq j} \mathrm{Cov}(\hat{f}_i, \hat{f}_j) \right) \\
&= \frac{1}{B^2} \bigl( B\sigma^2 + B(B-1)\rho\sigma^2 \bigr) \\
&= \frac{1}{B}\sigma^2 + \frac{B-1}{B}\rho\sigma^2 \\
&= \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2.
\end{aligned}
$$

При $\rho < 1$ дисперсия среднего строго меньше дисперсии одной модели: $\mathrm{Var}(\bar{f}) < \sigma^2$. С ростом $B$ второе слагаемое стремится к нулю, и дисперсия ансамбля асимптотически приближается к $\rho\sigma^2$. Таким образом, чем меньше корреляция между моделями, тем большее снижение дисперсии достигается; при $\rho = 0$ дисперсия уменьшается в $B$ раз.

#### Углублённый взгляд: асимптотическая дисперсия бэггинга и устойчивость алгоритма

Бутстреп‑ансамбль порождает модели, корреляция $\rho$ между которыми определяется **устойчивостью** базового алгоритма обучения. Для неустойчивых алгоритмов, таких как глубокие решающие деревья, малые изменения обучающей выборки (бутстреп) приводят к значительным вариациям предсказаний, поэтому корреляция между предсказаниями двух деревьев, обученных на разных бутстреп‑выборках, относительно невелика. Напротив, для устойчивых алгоритмов (например, $k$-ближайших соседей с большим $k$) бутстреп‑выборки почти не меняют модель, $\rho \approx 1$, и бэггинг не даёт выигрыша.

Формально, можно показать (см. Efron, Tibshirani, 1993; Buja, Stuetzle, 2006), что асимптотическая дисперсия бэггинг‑оценки связана с функцией влияния базового алгоритма. Для деревьев функция влияния неограничена, что и объясняет высокую эффективность бэггинга.

Бэггинг практически не изменяет смещение модели. Действительно, математическое ожидание предсказания ансамбля $\mathbb{E}[\bar{f}] = \frac{1}{B}\sum \mathbb{E}[\hat{f}_b] = \mu$ равно математическому ожиданию предсказания одной модели. Поскольку базовые модели (полные деревья) обладают низким смещением, ансамбль сохраняет это свойство, одновременно радикально снижая дисперсию.

### 3. Бэггинг для классификации

В задачах классификации с $K$ классами бэггинг‑ансамбль обычно использует один из двух механизмов агрегирования:

- **Жёсткое голосование (hard voting):** каждый классификатор отдаёт голос за один класс, итоговое решение — класс, набравший большинство голосов.
- **Мягкое голосование (soft voting):** каждый классификатор возвращает оценку вероятности $p_k^{(b)}(\mathbf{x})$ для каждого класса $k$; итоговая вероятность вычисляется как среднее $\bar{p}_k(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^B p_k^{(b)}(\mathbf{x})$, после чего выбирается класс с максимальной вероятностью.

Мягкое голосование обычно предпочтительнее, так как учитывает «уверенность» каждой модели, что часто приводит к более высокому качеству.

Если классификаторы независимы и каждый ошибается с вероятностью $p < 1/2$, то вероятность ошибки ансамбля при жёстком голосовании даётся биномиальным хвостом:

$$
P_{\text{err}} = \sum_{k = \lfloor B/2 \rfloor + 1}^{B} \binom{B}{k} p^k (1-p)^{B-k}.
$$

Эта величина убывает экспоненциально с ростом $B$ (при $p < 1/2$), что теоретически обосновывает эффективность ансамблевого голосования. На практике корреляция между моделями несколько замедляет этот процесс, но общая тенденция сохраняется.

### 4. Out‑of‑Bag (OOB) оценка

Каждая бутстреп‑выборка $\mathcal{D}_b^*$ оставляет без использования около $36{,}8\%$ объектов исходной выборки. Эти объекты образуют **Out‑of‑Bag (OOB)** набор для $b$-й модели. Для каждого объекта $i$ можно получить OOB‑предсказание, усреднив (или проголосовав) только по тем моделям, в обучении которых этот объект не участвовал:

$$
\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i),
$$

где $\mathcal{B}_i = \{b \mid (\mathbf{x}_i, y_i) \notin \mathcal{D}_b^*\}$.

OOB‑ошибка, вычисленная на всех объектах, является практически несмещённой оценкой ошибки обобщения, сопоставимой по точности с кросс‑валидацией, но получаемой «бесплатно» в процессе обучения ансамбля. В отличие от обычной обучающей ошибки, OOB‑ошибка не страдает от оптимистического смещения, так как для каждого объекта используются только модели, не видевшие его при обучении.

### 5. Код и эксперименты

Проиллюстрируем свойства бэггинга на наборе данных `breast_cancer`. В качестве базовой модели возьмём полное решающее дерево (`max_depth=None`), которое характеризуется низким смещением и высокой дисперсией.




In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# Параметры эксперимента
B_range = [1, 3, 5, 10, 20, 50, 100, 200]
train_acc = []
test_acc = []
oob_acc = []

for B in B_range:
    # Bagging с полными деревьями, bootstrap=True, oob_score=True
    bag = BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=0),
        n_estimators=B,
        bootstrap=True,
        oob_score=True,
        random_state=42,
        n_jobs=-1
    )
    bag.fit(X_train, y_train)
    train_acc.append(accuracy_score(y_train, bag.predict(X_train)))
    test_acc.append(accuracy_score(y_test, bag.predict(X_test)))
    oob_acc.append(bag.oob_score_)

# Визуализация
plt.figure(figsize=(8, 5))
plt.plot(B_range, train_acc, 'bo-', label='train')
plt.plot(B_range, test_acc, 'ro-', label='test')
plt.plot(B_range, oob_acc, 'g^--', label='OOB')
plt.xscale('log')
plt.xlabel('Число моделей B')
plt.ylabel('Accuracy')
plt.title('Бэггинг с полными решающими деревьями (breast_cancer)')
plt.legend()
plt.grid(True)
plt.show()


График демонстрирует характерное поведение: с увеличением числа моделей тестовая точность возрастает и насыщается, обучающая точность остаётся близкой к единице, а OOB‑оценка практически совпадает с тестовой. Это подтверждает, что бэггинг эффективно уменьшает дисперсию, не увеличивая смещение, и что OOB‑ошибка служит надёжным индикатором обобщающей способности.



### Вопросы для самопроверки

#### Для начинающих
1. Зачем нужен бэггинг и какую проблему одиночных решающих деревьев он решает?
2. Что такое бутстреп‑выборка и почему в ней примерно $63\%$ уникальных объектов?
3. Почему OOB‑ошибка считается почти несмещённой оценкой качества обобщения?
4. Как бэггинг влияет на смещение и дисперсию базовой модели?

#### Для профессионалов
1. Докажите формулу $\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ для среднего коррелированных случайных величин с равными дисперсиями и попарной корреляцией $\rho$.
2. Объясните, почему бэггинг не уменьшает смещение. Приведите математическое обоснование с использованием линейности математического ожидания.
3. Сравните OOB‑ошибку и кросс‑валидацию с точки зрения смещения и дисперсии. В каких случаях OOB‑ошибка может давать смещённую оценку?
4. Выведите вероятность ошибки ансамбля при жёстком голосовании для $B$ независимых классификаторов, каждый из которых ошибается с вероятностью $p$, и покажите, что она экспоненциально убывает с ростом $B$ при $p < 1/2$.


## 5.2. Случайный лес: декорреляция деревьев, важность признаков и теория

Бэггинг над решающими деревьями снижает дисперсию, усредняя предсказания многих моделей. Однако деревья, построенные по разным бутстреп‑выборкам, всё ещё могут быть сильно скоррелированы, если в данных присутствует несколько доминирующих признаков: каждое дерево будет использовать их в первых же расщеплениях, что приводит к схожей структуре и, как следствие, к высокой попарной корреляции предсказаний. Случайный лес (Random Forest), предложенный Брейманом (Breiman, 2001), решает эту проблему, вводя дополнительный источник случайности — при каждом расщеплении выбор признака ограничивается случайным подмножеством размера $m$ из полного набора $D$ признаков. Эта простая модификация радикально повышает точность ансамбля.

### 1. От бэггинга к случайному лесу

В случайном лесе сохраняется бутстреп‑агрегирование базовых деревьев. Главное отличие заключено в процедуре построения каждого дерева:

> Для каждого узла **перед расщеплением** из полного набора $D$ признаков случайным образом (без возвращения) отбирается подмножество размера $m$, и оптимальное расщепление ищется только среди этого подмножества.

Типичные рекомендации для выбора $m$, подтверждённые обширными экспериментами:

- **Классификация:** $m = \big\lfloor \sqrt{D} \big\rfloor$,
- **Регрессия:** $m = \big\lfloor D/3 \big\rfloor$.

Деревья строятся до максимальной глубины (без прунинга), часто вплоть до листьев, содержащих минимальное число объектов. Это сохраняет низкое смещение каждого отдельного дерева. Ансамблевое предсказание, как и в бэггинге, формируется усреднением (регрессия) или голосованием (классификация).

Математическая мотивация случайного выбора признаков непосредственно следует из формулы дисперсии среднего коррелированных случайных величин, выведенной ранее:

$$
\mathrm{Var}\big(\bar{f}\big) = \rho\,\sigma^2 + \frac{1-\rho}{B}\,\sigma^2.
$$

Если несколько признаков являются очень сильными, то бэггинг‑деревья будут похожи друг на друга ($\rho$ велико), и дисперсия ансамбля снижается незначительно. Ограничение числа кандидатов $m$ **принудительно** заставляет деревья использовать различные признаки, что уменьшает среднюю попарную корреляцию $\rho$. Пусть корреляция между предсказаниями двух деревьев, обученных на бутстреп‑выборках, в бэггинге равна $\rho_{\text{bag}}$, а в случайном лесе $\rho_{\text{RF}}$. Тогда при $m < D$ выполняется $\rho_{\text{RF}} < \rho_{\text{bag}}$, а значит, и дисперсия ансамбля $\mathrm{Var}_{\text{RF}} < \mathrm{Var}_{\text{bag}}$. При $m = D$ случайный лес вырождается в бэггинг.

Вместе с тем, чрезмерное уменьшение $m$ увеличивает смещение каждого отдельного дерева, потому что важные признаки могут быть исключены из рассмотрения во многих узлах, что ухудшает качество подгонки. Оптимальное значение $m$ находится как компромисс между декорреляцией (снижение дисперсии ансамбля) и сохранением точности отдельных деревьев (контроль смещения).

### 2. Анализ смещения и дисперсии для случайного леса

Проведём более детальный анализ компромисса «смещение–дисперсия» для случайного леса. Обозначим через $\hat{f}_b(\mathbf{x})$ предсказание $b$-го дерева, а через $\bar{f}(\mathbf{x}) = \frac{1}{B}\sum_{b=1}^B \hat{f}_b(\mathbf{x})$ — предсказание ансамбля. Математическое ожидание ансамбля равно

$$
\mathbb{E}\big[\bar{f}(\mathbf{x})\big] = \frac{1}{B}\sum_{b=1}^B \mathbb{E}\big[\hat{f}_b(\mathbf{x})\big] = \mu(\mathbf{x}),
$$

поскольку все деревья строятся по одному и тому же вероятностному механизму. Таким образом, ансамбль практически не меняет смещение по сравнению с одним деревом.

Дисперсия каждого отдельного дерева зависит от $m$: чем меньше $m$, тем меньше информации используется при расщеплении, и тем выше дисперсия предсказаний одного дерева (как функции обучающей выборки). Однако усреднение по ансамблю преобразует индивидуальную дисперсию $\sigma^2$ и попарную корреляцию $\rho$ в общую дисперсию согласно формуле $\rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$. Уменьшение $m$ снижает $\rho$ (положительный эффект) и одновременно может увеличить $\sigma^2$ (отрицательный эффект). При достаточно большом числе деревьев $B$ определяющим становится первое слагаемое $\rho\sigma^2$, поэтому снижение корреляции является приоритетной целью, даже ценой умеренного роста индивидуальной дисперсии. Этим объясняется, почему стандартные эвристики $m = \sqrt{D}$ или $m = D/3$ оказываются близкими к оптимальным для широкого круга задач.

### 3. Out‑of‑Bag (OOB) оценка в случайном лесе

Каждое дерево случайного леса обучается на своей бутстреп‑выборке, содержащей в среднем около $63{,}2\%$ уникальных объектов исходной выборки. Оставшиеся объекты, не попавшие в бутстреп‑выборку, образуют **Out‑of‑Bag** (OOB) набор для этого дерева. Для каждого объекта $i$ можно построить OOB‑предсказание, усреднив (или проголосовав) только по тем деревьям, в обучении которых этот объект не участвовал:

$$
\hat{y}_i^{\text{OOB}} = \frac{1}{|\mathcal{B}_i|} \sum_{b \in \mathcal{B}_i} \hat{f}_b(\mathbf{x}_i),
$$

где $\mathcal{B}_i = \{b \mid (\mathbf{x}_i, y_i) \notin \mathcal{D}_b^*\}$.

OOB‑ошибка, вычисленная на всех объектах, является практически несмещённой оценкой ошибки обобщения случайного леса. В отличие от обучающей ошибки, OOB‑ошибка не страдает от оптимистического смещения, так как для каждого объекта используются только модели, не видевшие его при обучении. OOB‑оценка получается «бесплатно» в процессе построения леса и часто используется для настройки гиперпараметров без выделения валидационной выборки.

### 4. Важность признаков в случайном лесе

Одно из главных достоинств случайного леса — возможность оценить вклад каждого признака в предсказание. Вводится две основные меры важности: основанная на уменьшении impurity (MDI) и основанная на перестановке (permutation importance).

#### 4.1. Уменьшение impurity (Mean Decrease in Impurity, MDI)

Для каждого признака $j$ суммируется уменьшение критерия impurity (например, Gini или энтропия) по всем узлам всех деревьев, в которых признак $j$ использовался для расщепления. Уменьшение взвешивается числом обучающих объектов, прошедших через узел. Формально, для одного дерева $T$:

$$
\text{FI}_j(T) = \sum_{t \in T: \text{split}(t) = j} \frac{m_t}{n} \,\Delta I_t,
$$

где $m_t$ — число объектов в узле $t$, а

$$
\Delta I_t = I_t - \frac{m_{t,L}}{m_t} I_{t,L} - \frac{m_{t,R}}{m_t} I_{t,R}
$$

— уменьшение impurity в узле $t$. Итоговая MDI‑важность получается усреднением $\text{FI}_j(T)$ по всем деревьям леса.

MDI вычислительно дёшева, так как все величины $\Delta I_t$ и $m_t$ известны из процесса обучения. Однако MDI обладает известным смещением в пользу признаков с большим числом уникальных значений или категорий. Такие признаки предоставляют больше возможных разбиений, и даже если они не связаны с целевой переменной, жадный алгоритм может выбрать одно из них, создавая ложное уменьшение impurity. Кроме того, для признаков, имеющих более широкий диапазон значений, выше вероятность того, что случайно выбранный порог даст заметное уменьшение impurity.

#### 4.2. Permutation Importance (важность по перестановке)

Идея, предложенная Брейманом (2001), состоит в измерении падения качества предсказаний при разрушении связи между признаком и целевой переменной. Для каждого признака $j$ его значения случайным образом перемешиваются по наблюдениям (пермутация), после чего вычисляется ошибка модели на изменённых данных. Важность определяется как средняя разность между ошибкой на перемешанных данных и исходной ошибкой:

$$
\text{FI}_j = \frac{1}{R}\sum_{r=1}^R \bigl[ \text{Err}(\tilde{X}_j^{(r)}) - \text{Err}_{\text{orig}} \bigr],
$$

где $\tilde{X}_j^{(r)}$ — матрица признаков с $r$-й случайной перестановкой $j$-го столбца, а $R$ — число повторных перестановок.

Ошибка обычно измеряется как разность в accuracy (для классификации) или в MSE (для регрессии). В качестве исходной ошибки $\text{Err}_{\text{orig}}$ берётся ошибка модели на исходных данных (обычно OOB‑ошибка или ошибка на отложенной выборке).

Главное достоинство permutation importance — независимость от структуры модели и отсутствие смещения в сторону категорийных признаков. Однако для сильно коррелированных признаков перестановка одного из них не всегда полностью разрушает информацию (дублирующий признак может частично компенсировать), что иногда приводит к заниженным оценкам важности каждого из коррелированной группы. Современные модификации, такие как **Conditional Permutation Importance**, переставляют признак условно на другие признаки, что позволяет более честно оценивать вклад в присутствии коллинеарности.

#### 4.3. Сравнение MDI и Permutation Importance

Две меры часто коррелируют, но могут давать различный порядок значимости. MDI склонна завышать важность многозначных и категориальных признаков, тогда как permutation importance не обладает этим недостатком. На практике рекомендуется анализировать обе меры, а при противоречиях — больше доверять permutation importance или использовать её условную версию.

#### Углублённый взгляд: связь MDI с объяснённой дисперсией в регрессии

В регрессионном случайном лесе, использующем MSE в качестве impurity, суммарное уменьшение MSE по всем узлам леса обладает прозрачной вероятностной интерпретацией. Для одного дерева сумма $\sum_t \frac{m_t}{n} \Delta \text{MSE}_t$ равна разности между дисперсией отклика в корневом узле и средневзвешенной дисперсией в листьях — то есть доле дисперсии, «объяснённой» разбиениями. Усредняя по лесу, получаем, что MDI признака $j$ асимптотически (с ростом числа деревьев) оценивает вклад этого признака в объяснённую дисперсию. Более формально, рассматривая случайный лес как адаптивную оценку функции регрессии, MDI можно связать с функциональным разложением ANOVA и оценками чувствительности (см. работы Штробля и др., 2007; Грёмпе, 2009). Это делает MDI особенно интерпретируемой мерой для регрессии. Для классификации интерпретация через уменьшение энтропии менее прямая, но сохраняет аналогичный смысл «уменьшения неопределённости» целевой переменной.

### 5. Эксперименты

Продемонстрируем свойства случайного леса на наборе данных `breast_cancer`. Обучим лес из 500 деревьев с рекомендованным параметром $m = \lfloor\sqrt{D}\rfloor$, сравним с бэггингом и одиночным деревом, исследуем важность признаков и влияние $m$ на качество. Также воспользуемся встроенной OOB‑оценкой.



In [ ]:
# @title Случайный лес: обучение, OOB, важность признаков и влияние m
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import BaggingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score

# Загрузка данных
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

D = X.shape[1]
m_default = int(np.sqrt(D))
print(f"Число признаков: {D}, рекомендованное m: {m_default}")

# Обучение моделей (для леса включаем OOB-подсчёт)
rf = RandomForestClassifier(n_estimators=500, max_features=m_default,
                            random_state=42, n_jobs=-1, oob_score=True)
rf.fit(X_train, y_train)

bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(random_state=0),
    n_estimators=500, random_state=42, n_jobs=-1)
bag.fit(X_train, y_train)

tree = DecisionTreeClassifier(random_state=0)
tree.fit(X_train, y_train)

# Сравнение точности и OOB
models = {'Одиночное дерево': tree, 'Бэггинг (500)': bag,
          'Случайный лес (500)': rf}
for name, model in models.items():
    acc = accuracy_score(y_test, model.predict(X_test))
    oob_str = f", OOB = {model.oob_score_:.4f}" if hasattr(model, 'oob_score_') else ""
    print(f"{name}: test accuracy = {acc:.4f}{oob_str}")


Ожидается, что случайный лес превзойдёт и одиночное дерево, и бэггинг благодаря дополнительной декорреляции, а OOB‑оценка будет близка к тестовой точности.

#### Важность признаков (MDI и Permutation)

Извлечём MDI из обученного леса и рассчитаем permutation importance на тестовой выборке. Для сравнения построим горизонтальные диаграммы топ‑10 признаков и scatter‑plot двух мер.




In [ ]:
# MDI (встроенная важность)
mdi_importances = rf.feature_importances_
indices_mdi = np.argsort(mdi_importances)[::-1]

# Permutation importance на тесте
perm_result = permutation_importance(rf, X_test, y_test, n_repeats=10,
                                     random_state=42, n_jobs=-1)
perm_importances = perm_result.importances_mean
indices_perm = np.argsort(perm_importances)[::-1]

# Визуализация топ-10 признаков по каждой мере
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(range(10), mdi_importances[indices_mdi][:10][::-1])
axes[0].set_yticks(range(10))
axes[0].set_yticklabels([data.feature_names[i] for i in indices_mdi[:10][::-1]])
axes[0].set_title('MDI importance (топ-10)')
axes[0].set_xlabel('Уменьшение impurity')

axes[1].barh(range(10), perm_importances[indices_perm][:10][::-1])
axes[1].set_yticks(range(10))
axes[1].set_yticklabels([data.feature_names[i] for i in indices_perm[:10][::-1]])
axes[1].set_title('Permutation importance (топ-10)')
axes[1].set_xlabel('Падение accuracy')

plt.tight_layout()
plt.show()

# Сравнение двух мер (scatter plot всех признаков)
plt.figure(figsize=(7,6))
plt.scatter(mdi_importances, perm_importances)
plt.xlabel('MDI importance')
plt.ylabel('Permutation importance')
plt.title('MDI vs Permutation importance')
for i in range(D):
    if mdi_importances[i] > 0.05 or perm_importances[i] > 0.02:
        plt.annotate(data.feature_names[i], (mdi_importances[i], perm_importances[i]),
                     fontsize=8, alpha=0.7)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


Scatter‑plot позволяет заметить признаки, для которых MDI велика, а permutation importance — мала (потенциальное смещение MDI).

#### Зависимость точности от $m$

Зафиксируем $B=500$ и проварьируем `max_features` от 1 до $D$. Для каждого $m$ зафиксируем также OOB‑ошибку.




In [ ]:
m_values = np.arange(1, D+1)
test_acc = []
oob_acc = []
for m in m_values:
    rf_temp = RandomForestClassifier(n_estimators=500, max_features=m,
                                     random_state=42, n_jobs=-1, oob_score=True)
    rf_temp.fit(X_train, y_train)
    test_acc.append(accuracy_score(y_test, rf_temp.predict(X_test)))
    oob_acc.append(rf_temp.oob_score_)

plt.figure(figsize=(8,5))
plt.plot(m_values, test_acc, 'o-', label='Test accuracy')
plt.plot(m_values, oob_acc, 's-', label='OOB accuracy')
plt.axvline(m_default, color='r', linestyle='--', label=f'default m={m_default}')
plt.xlabel('m (max_features)')
plt.ylabel('Accuracy')
plt.title('Влияние числа случайных признаков на точность случайного леса')
plt.legend()
plt.grid()
plt.show()


График демонстрирует, что точность растёт с увеличением $m$, достигает плато в районе рекомендованного значения, после чего может незначительно снижаться. OOB‑оценка практически повторяет поведение тестовой, подтверждая свою пригодность для настройки гиперпараметров без выделения валидационной выборки.

#### Стабилизация ошибки с ростом числа деревьев

В отличие от многих других алгоритмов, случайный лес **не переобучается** при увеличении числа деревьев $B$: тестовая ошибка монотонно убывает и стабилизируется около некоторого асимптотического уровня. Проиллюстрируем это, зафиксировав $m = m_{\text{default}}$ и варьируя $B$.




In [ ]:
B_values = np.logspace(1, 3, 15, dtype=int)  # от 10 до 1000 с логарифмическим шагом
test_err = []
oob_err = []
for B in B_values:
    rf_temp = RandomForestClassifier(n_estimators=B, max_features=m_default,
                                     random_state=42, n_jobs=-1, oob_score=True)
    rf_temp.fit(X_train, y_train)
    test_err.append(1 - accuracy_score(y_test, rf_temp.predict(X_test)))
    oob_err.append(1 - rf_temp.oob_score_)

plt.figure(figsize=(8,5))
plt.plot(B_values, test_err, 'o-', label='Test error')
plt.plot(B_values, oob_err, 's-', label='OOB error')
plt.xscale('log')
plt.xlabel('Число деревьев B')
plt.ylabel('Ошибка')
plt.title('Сходимость ошибки случайного леса с ростом B')
plt.legend()
plt.grid()
plt.show()


Как видно, ошибка стабилизируется уже при нескольких десятках деревьев; дальнейшее увеличение $B$ лишь незначительно улучшает результат, но не приводит к переобучению.


### 6. Числовая иллюстрация построения случайного леса

Для закрепления теории рассмотрим сквозной числовой пример, демонстрирующий ключевые этапы работы случайного леса: формирование бутстреп‑выборки, построение одного дерева со случайным подмножеством признаков, вычисление impurity, выбор порога, агрегирование предсказаний и оценку важности признаков.

**Данные.**  
Пусть обучающая выборка содержит $n = 6$ объектов с двумя числовыми признаками $x_1$ и $x_2$ и бинарной целевой переменной $y \in \{0, 1\}$:

| $i$ | $x_1$ | $x_2$ | $y$ |
|-----|-------|-------|-----|
| 1   | 2     | 3     | 0   |
| 2   | 1     | 2     | 0   |
| 3   | 3     | 1     | 0   |
| 4   | 5     | 4     | 1   |
| 5   | 6     | 5     | 1   |
| 6   | 4     | 6     | 1   |

Это линейно разделимые данные: класс $0$ сосредоточен в левом нижнем углу, класс $1$ — в правом верхнем.

**Параметры леса (для иллюстрации).**  
Число деревьев $B = 2$, число случайно отбираемых признаков в каждом узле $m = 1$ (при $D = 2$ по формуле $\lfloor\sqrt{2}\rfloor = 1$). Деревья строятся до глубины $2$ без ограничения на минимальное число объектов в листе.

#### 6.1. Первое дерево

**Бутстреп‑выборка.**  
Из исходной выборки объёмом $n = 6$ случайным образом (с возвращением) сформируем бутстреп‑выборку. Допустим, выпали индексы $\{1, 2, 2, 4, 5, 6\}$. Тогда данные для первого дерева:

| $i$ (в бутстрепе) | $x_1$ | $x_2$ | $y$ |
|-------------------|-------|-------|-----|
| 1 (объект 1)      | 2     | 3     | 0   |
| 2 (объект 2)      | 1     | 2     | 0   |
| 3 (объект 2)      | 1     | 2     | 0   |
| 4 (объект 4)      | 5     | 4     | 1   |
| 5 (объект 5)      | 6     | 5     | 1   |
| 6 (объект 6)      | 4     | 6     | 1   |

Объект 3 исходной выборки не попал в бутстреп, а объект 2 дублирован.

**Корневой узел.**  
В корневом узле находятся все 6 объектов. Оценим распределение классов: три объекта класса $0$, три объекта класса $1$. Impurity по индексу Джини:

$$
I_{\text{root}} = 1 - \left(\frac{3}{6}\right)^2 - \left(\frac{3}{6}\right)^2 = 1 - \frac{1}{4} - \frac{1}{4} = 0.5.
$$

Согласно $m = 1$, случайно отбираем один признак из двух. Пусть выбран $x_1$. Ищем оптимальный порог $\theta$ для разбиения. Уникальные значения $x_1$ в узле: $\{1, 2, 4, 5, 6\}$ (с учётом дубликата $1$). Пороги-кандидаты — середины между соседними значениями: $\{1.5, 3.0, 4.5, 5.5\}$.

Для каждого порога вычисляем взвешенную impurity дочерних узлов:

- $\theta = 1.5$: левый узел — объекты с $x_1 \le 1.5$ (три объекта: все $y=0$), правый узел — остальные (три объекта: все $y=1$). Тогда $I_{\text{left}} = 0$, $I_{\text{right}} = 0$, взвешенная impurity $= 0$. Уменьшение impurity: $\Delta I = 0.5 - 0 = 0.5$.

- $\theta = 3.0$: левый узел — три объекта класса $0$ (все), правый — три объекта класса $1$ (все). Аналогично $\Delta I = 0.5$.

- $\theta = 4.5$: левый узел — три объекта класса $0$ и один объект класса $1$ (объект 4 с $x_1=5$ попадает в правый). Тогда в левом узле: класс $0$ — $3$, класс $1$ — $0$? Проверим: при $\theta = 4.5$ объекты с $x_1 \le 4.5$ — это объекты 1,2,2,4? У объекта 4 $x_1=5 > 4.5$, значит он в правом. Левый: объекты 1,2,2 — все $y=0$, impurity $0$. Правый: объекты 4,5,6 — все $y=1$, impurity $0$. И снова $\Delta I = 0.5$.

- $\theta = 5.5$: левый узел — объекты 1,2,2,4,6? Нет, у объекта 6 $x_1=4 \le 5.5$, он в левом. Тогда левый: четыре объекта (три класса $0$, один класса $1$ — объект 4 с $y=1$). Правый: объект 5 ($y=1$). Тогда $I_{\text{left}} = 1 - (3/4)^2 - (1/4)^2 = 1 - 9/16 - 1/16 = 6/16 = 0.375$, $I_{\text{right}} = 0$ (чистый). Взвешенная impurity $= \frac{4}{6} \cdot 0.375 + \frac{2}{6} \cdot 0 = 0.25$. $\Delta I = 0.5 - 0.25 = 0.25$.

Максимальное $\Delta I = 0.5$ достигается при порогах $1.5, 3.0, 4.5$. Выберем, например, $\theta = 3.0$. Тогда левое поддерево ($x_1 \le 3$) содержит объекты $\{1,2,2\}$ (все $y=0$), правое ($x_1 > 3$) — объекты $\{4,5,6\}$ (все $y=1$). Оба дочерних узла чисты, impurity равна нулю, дальнейшие разбиения не требуются — они становятся листьями с предсказаниями $0$ и $1$ соответственно. Глубина дерева достигла $1$, что меньше разрешённой $2$, но узлы уже чисты.

Итак, первое дерево: корень — проверка $x_1 \le 3.0$, левый лист — класс $0$, правый лист — класс $1$. Признак $x_2$ в этом дереве не использовался.

#### 6.2. Второе дерево

Сгенерируем другую бутстреп‑выборку. Пусть выпали индексы $\{3, 3, 4, 4, 5, 6\}$ (объекты 3 и 4 повторены, объекты 1 и 2 отсутствуют). Данные:

| $i$ (в бутстрепе) | $x_1$ | $x_2$ | $y$ |
|-------------------|-------|-------|-----|
| 1 (объект 3)      | 3     | 1     | 0   |
| 2 (объект 3)      | 3     | 1     | 0   |
| 3 (объект 4)      | 5     | 4     | 1   |
| 4 (объект 4)      | 5     | 4     | 1   |
| 5 (объект 5)      | 6     | 5     | 1   |
| 6 (объект 6)      | 4     | 6     | 1   |

Корневой узел: два объекта класса $0$, четыре объекта класса $1$. Impurity:

$$
I_{\text{root}} = 1 - \left(\frac{2}{6}\right)^2 - \left(\frac{4}{6}\right)^2 = 1 - \frac{4}{36} - \frac{16}{36} = \frac{16}{36} \approx 0.444.
$$

Снова $m = 1$, случайно выбираем один признак. Допустим, выпал $x_2$. Уникальные значения $x_2$: $\{1, 4, 5, 6\}$. Пороги-кандидаты: $\{2.5, 4.5, 5.5\}$.

- $\theta = 2.5$: левый узел — объекты с $x_2 \le 2.5$ (два объекта класса $0$), правый — четыре объекта класса $1$. Оба чисты. $\Delta I = 0.444 - 0 = 0.444$.
- $\theta = 4.5$: левый — два объекта класса $0$ и два объекта класса $1$ (объекты 3,3 с $y=0$ и 4,4 с $y=1$). Тогда $I_{\text{left}} = 1 - (2/4)^2 - (2/4)^2 = 0.5$. Правый — два объекта класса $1$ (чист, impurity $0$). Взвешенная impurity $= \frac{4}{6} \cdot 0.5 + \frac{2}{6} \cdot 0 \approx 0.333$. $\Delta I = 0.444 - 0.333 = 0.111$.
- $\theta = 5.5$: левый — два объекта класса $0$ и четыре объекта класса $1$? Проверим: при $\theta = 5.5$ объекты с $x_2 \le 5.5$ — это объекты 3,3,4,4,5 (объект 6 с $x_2=6$ не входит). Тогда левый: два класса $0$, три класса $1$. $I_{\text{left}} = 1 - (2/5)^2 - (3/5)^2 = 1 - 4/25 - 9/25 = 12/25 = 0.48$. Правый: один объект класса $1$ (чист). Взвешенная impurity $= \frac{5}{6} \cdot 0.48 + \frac{1}{6} \cdot 0 = 0.4$. $\Delta I = 0.444 - 0.4 = 0.044$.

Максимальное $\Delta I$ даёт порог $2.5$, создающий два чистых дочерних узла. Выбираем его. Левое поддерево ($x_2 \le 2.5$) — объекты $\{3,3\}$ (класс $0$), правое ($x_2 > 2.5$) — объекты $\{4,4,5,6\}$ (класс $1$). Оба листа чисты.

Второе дерево: корень проверяет $x_2 \le 2.5$, левый лист — класс $0$, правый — класс $1$. Признак $x_1$ не использовался.

#### 6.3. Агрегирование и предсказание

Теперь у нас есть ансамбль из двух деревьев. Как классифицировать новый объект, например, $\mathbf{x}_* = (3.5, 3.5)$?

- Первое дерево: проверяет $x_1 \le 3.0$. У объекта $x_1 = 3.5 > 3.0$, поэтому он попадает в правый лист, предсказание класса $1$.
- Второе дерево: проверяет $x_2 \le 2.5$. У объекта $x_2 = 3.5 > 2.5$, поэтому правое поддерево, предсказание класса $1$.

Оба дерева голосуют за класс $1$. Итоговое предсказание — класс $1$.

Рассмотрим другой объект $\mathbf{x}_* = (2.0, 1.5)$:

- Первое дерево: $x_1 = 2.0 \le 3.0$, левый лист, предсказание $0$.
- Второе дерево: $x_2 = 1.5 \le 2.5$, левый лист, предсказание $0$.

Итог — класс $0$.

Объект $\mathbf{x}_* = (4.0, 2.0)$:

- Первое дерево: $x_1 = 4.0 > 3.0$, предсказание $1$.
- Второе дерево: $x_2 = 2.0 \le 2.5$, левый лист, предсказание $0$.

Здесь голоса разделились: одно дерево за класс $1$, другое за класс $0$. При жёстком голосовании с нечётным числом деревьев был бы выбран класс большинства; при двух деревьях можно предусмотреть правило случайного выбора или опираться на мягкое голосование (усреднение вероятностей). В реальных лесах с сотнями деревьев подобные коллизии исчезают.

#### 6.4. Важность признаков

Оценим MDI и Permutation Importance на нашем ансамбле из двух деревьев, используя исходную обучающую выборку (а не бутстреп‑выборки).

**MDI (Mean Decrease in Impurity).**  
Вспомним, что MDI признака $j$ равна сумме взвешенных уменьшений impurity по всем узлам всех деревьев, где признак использовался для разбиения. Для первого дерева: признак $x_1$ использовался в корневом узле с $m_t = 6$, $\Delta I = 0.5$. Вклад в MDI: $\frac{6}{6} \cdot 0.5 = 0.5$ (нормировка на исходный размер выборки $n=6$). Для второго дерева: признак $x_2$ в корне с $m_t = 6$, $\Delta I \approx 0.444$, вклад $0.444$. Итоговые MDI (усреднённые по двум деревьям): $x_1 \mapsto 0.25$, $x_2 \mapsto 0.222$. (При большем числе деревьев значения усредняются более полно.)

**Permutation Importance.**  
Оценим важность, измеряя падение точности на исходных данных после случайной перестановки значений признака. Исходная точность ансамбля на обучающей выборке (все 6 объектов) составляет $100\%$, так как каждое дерево безошибочно классифицирует свои бутстреп‑выборки, а вместе они правильно предсказывают все объекты (проверьте: объект 3 не был в первом дереве, но второе дерево его классифицирует верно). Перемешаем случайно значения признака $x_1$ между объектами. Допустим, после перемешивания столбец $x_1$ стал $(4, 5, 2, 1, 6, 3)$ (случайная перестановка). Прогоним через оба дерева и зафиксируем ошибку. Повторим перестановку несколько раз и усредним разность между полученной ошибкой и исходной. Аналогично для $x_2$. Поскольку в нашем маленьком примере деревья идеально разделяют классы, любая перестановка, скорее всего, приведёт к ошибке. Точные числа можно получить, выполнив несколько итераций вручную или в коде, но принципиальный результат: оба признака получат положительную важность, причём более важный признак (тот, который чаще используется в корневых разбиениях) будет иметь большее значение.

Этот пример наглядно демонстрирует, как случайный лес комбинирует деревья, использующие разные признаки, и как вычисляются меры важности. В реальных приложениях с сотнями деревьев и признаков описанные шаги полностью автоматизированы, но понимание их механики позволяет осознанно интерпретировать результаты и настраивать модель.


### 6. Итог и рекомендации

Случайный лес — один из наиболее надёжных и универсальных алгоритмов машинного обучения. Его основные достоинства:

- автоматическое снижение дисперсии без увеличения смещения;
- встроенная оценка обобщающей способности через OOB;
- информативные меры важности признаков (MDI и permutation);
- устойчивость к выбросам, шуму и пропущенным данным;
- минимальная настройка гиперпараметров.

При практическом использовании рекомендуется:

- Начинать со стандартных значений $m = \sqrt{D}$ (классификация) или $m = D/3$ (регрессия).
- Число деревьев $B$ брать порядка нескольких сотен; контролировать сходимость можно по OOB‑ошибке.
- Анализировать важность признаков, применяя как MDI, так и permutation importance; при наличии сильно коррелированных признаков отдавать предпочтение условной permutation importance.
- Сравнивать случайный лес с градиентным бустингом, который при правильной настройке может давать более высокую точность, но более чувствителен к гиперпараметрам и склонен к переобучению.


### Вопросы для самопроверки

#### Для начинающих
1. Чем случайный лес отличается от обычного бэггинга над решающими деревьями?
2. Зачем при построении деревьев в случайном лесе используется случайное подмножество признаков?
3. Как измерить важность признака в случайном лесе? Опишите идею MDI и Permutation Importance.
4. Что лучше: много глубоких деревьев или много неглубоких деревьев в случайном лесе, и почему?
5. Что такое OOB‑оценка и почему она полезна?

#### Для профессионалов
1. Выведите формулу дисперсии ансамбля для среднего коррелированных предсказаний и покажите, как выбор $m$ влияет на корреляцию $\rho$ и, следовательно, на итоговую дисперсию.
2. Объясните причину смещения MDI в пользу признаков с большим числом категорий. Предложите способ коррекции этого смещения.
3. Сравните permutation importance и SHAP (SHapley Additive exPlanations) с точки зрения учёта корреляций между признаками и вычислительной сложности.
4. Как теоретически выбор $m$ влияет на смещение и дисперсию каждого отдельного дерева и всего ансамбля? Опишите компромисс «смещение–дисперсия» в контексте случайного леса.
5. Докажите, что с увеличением числа деревьев $B$ ошибка случайного леса сходится к пределу, который не зависит от $B$, и объясните, почему переобучения не происходит.

## Градиентный бустинг: функциональный градиентный спуск и вывод для произвольных функций потерь

В отличие от бэггинга и случайного леса, которые строят ансамбль независимых моделей и усредняют их предсказания, градиентный бустинг (Friedman, 2001) формирует ансамбль последовательно: каждая следующая модель исправляет ошибки предыдущих. Математически этот процесс интерпретируется как функциональный градиентный спуск — оптимизация функции предсказания в функциональном пространстве.

### 1. Идея градиентного бустинга как функциональной оптимизации

Рассмотрим задачу обучения с учителем: по выборке $\mathcal{D} = \{(\mathbf{x}_i, y_i)\}_{i=1}^n$ требуется найти функцию $F: \mathbb{R}^D \to \mathbb{R}$, минимизирующую эмпирический риск

$$
\mathcal{L}(F) = \sum_{i=1}^n L(y_i, F(\mathbf{x}_i)),
$$

где $L(y, F)$ — дифференцируемая по $F$ функция потерь. Мы ищем $F$ в виде аддитивной композиции $M$ слабых моделей (обычно неглубоких решающих деревьев):

$$
F_M(\mathbf{x}) = \sum_{m=0}^M \nu \, h_m(\mathbf{x}),
$$

$h_0(\mathbf{x})$ — начальное приближение (константа), $\nu > 0$ — темп обучения (learning rate). Ключевая идея: строить $h_m$ последовательно, на каждом шаге выбирая такую функцию, которая при добавлении к $F_{m-1}$ максимально уменьшает $\mathcal{L}$. Функциональный градиент определяет направление наискорейшего возрастания $\mathcal{L}$ в точке $F_{m-1}$; это функция

$$
\mathbf{x} \mapsto \left.\frac{\partial L(y, F(\mathbf{x}))}{\partial F}\right|_{F = F_{m-1}(\mathbf{x})}.
$$

Чтобы уменьшить потери, нужно добавить к $F_{m-1}$ функцию, пропорциональную отрицательному градиенту. Поскольку мы ограничены классом $\mathcal{H}$ (например, деревья фиксированной глубины), новую модель $h_m$ обучают так, чтобы она наилучшим образом приближала компоненты антиградиента на обучающих объектах.

### 2. Вывод псевдо‑остатков (обобщённый градиент)

Разложим $L(y_i, F_{m-1}(\mathbf{x}_i) + f(\mathbf{x}_i))$ по Тейлору первого порядка относительно $f$:

$$
L(y_i, F_{m-1}(\mathbf{x}_i) + f(\mathbf{x}_i)) \approx L(y_i, F_{m-1}(\mathbf{x}_i)) + g_i \, f(\mathbf{x}_i),
$$

где

$$
g_i = \left. \frac{\partial L(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}.
$$

Слагаемое $g_i f(\mathbf{x}_i)$ линейно по $f$. Чтобы максимально уменьшить лосс, следует выбрать $f$ пропорциональной вектору $(-g_1,\dots,-g_n)$ на обучающей выборке. Поэтому в качестве целевых значений для обучения $h_m$ принимают **псевдо‑остатки**

$$
r_{im} = -g_i.
$$

Рассмотрим основные функции потерь.

**Квадратичная ошибка (регрессия).** $L(y, F) = \frac{1}{2}(y - F)^2$. Тогда

$$
g_i = \frac{\partial L}{\partial F} = -(y_i - F_{m-1}(\mathbf{x}_i)),
$$

откуда $r_{im} = y_i - F_{m-1}(\mathbf{x}_i)$ — обычные остатки текущей модели.

**Log‑loss для бинарной классификации с метками $y \in \{-1, +1\}$.**  
Функция потерь имеет вид

$$
L(y, F) = \log\bigl(1 + \exp(-2yF)\bigr).
$$

Градиент по $F$:

$$
\begin{aligned}
\frac{\partial L}{\partial F}
&= \frac{1}{1 + \exp(-2yF)} \cdot \frac{\partial}{\partial F}\bigl(1 + \exp(-2yF)\bigr) \\
&= \frac{1}{1 + \exp(-2yF)} \cdot \bigl(-2y \exp(-2yF)\bigr) \\
&= \frac{-2y \exp(-2yF)}{1 + \exp(-2yF)}
= \frac{-2y}{1 + \exp(2yF)}.
\end{aligned}
$$

Следовательно,

$$
g_i = \frac{-2y_i}{1 + \exp\bigl(2y_i F_{m-1}(\mathbf{x}_i)\bigr)}, \qquad
r_{im} = -g_i = \frac{2y_i}{1 + \exp\bigl(2y_i F_{m-1}(\mathbf{x}_i)\bigr)}.
$$

Интерпретация: величина $y_i F_{m-1}(\mathbf{x}_i)$ — отступ (margin). Если отступ велик и положителен, знаменатель велик, $r_{im}$ мал — модель уверена в правильном ответе, корректировка почти не требуется. При отрицательном отступе $r_{im} \approx 2y_i$, что задаёт направление исправления.

**Логистическая регрессия с $y \in \{0, 1\}$ и кросс‑энтропийной функцией потерь.**  
Используется

$$
L(y, F) = -\bigl[y \log \sigma(F) + (1-y) \log(1 - \sigma(F))\bigr], \qquad \sigma(F) = \frac{1}{1 + e^{-F}}.
$$

Производная по $F$:

$$
\frac{\partial L}{\partial F} = \sigma(F) - y.
$$

Поэтому

$$
g_i = \sigma(F_{m-1}(\mathbf{x}_i)) - y_i, \qquad
r_{im} = y_i - \sigma(F_{m-1}(\mathbf{x}_i)),
$$

то есть разность между истинной меткой и предсказанной вероятностью класса 1.

Во всех случаях $h_m$ обучается как регрессионная модель, предсказывающая псевдо‑остатки $r_{im}$ по $\mathbf{x}_i$. После этого новая модель добавляется с весом $\nu$:

$$
F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu \, h_m(\mathbf{x}).
$$

### 3. Выбор оптимального шага $\nu$ (learning rate)

Параметр $\nu$ управляет величиной шага функционального градиентного спуска. Малый фиксированный $\nu$ (shrinkage) замедляет обучение, но улучшает обобщающую способность, сглаживая вклад отдельных моделей. В пределе $\nu \to 0$ и $M \to \infty$ при $\nu M = \text{const}$ процесс аппроксимирует непрерывный градиентный спуск в функциональном пространстве, обладая регуляризирующим эффектом, сходным с $L_2$-штрафом.

Альтернативно, для каждого $m$ можно подбирать $\nu_m$ путём одномерной оптимизации (line search):

$$
\nu_m = \arg\min_{\nu} \sum_{i=1}^n L\bigl(y_i, F_{m-1}(\mathbf{x}_i) + \nu \, h_m(\mathbf{x}_i)\bigr).
$$

Такой подход используется, например, в классическом алгоритме AdaBoost для экспоненциальной функции потерь, где оптимальный вес имеет аналитическое выражение. В общем случае применяют численную оптимизацию (метод Ньютона или просто фиксируют малое $\nu$, компенсируя большим числом итераций).

### 4. Алгоритм градиентного бустинга (Generic Gradient Boosting)

Формально алгоритм выглядит следующим образом.

1. **Инициализация:**  
   $F_0(\mathbf{x}) = \arg\min_\gamma \sum_{i=1}^n L(y_i, \gamma)$.  
   Для квадратичной ошибки это среднее значение $y_i$; для логистической регрессии — логарифм отношения шансов $\log\frac{\sum y_i}{\sum (1-y_i)}$.

2. **Для $m = 1$ до $M$:**  
   1. Вычислить псевдо‑остатки:  
      $$ r_{im} = -\left. \frac{\partial L(y_i, F)}{\partial F} \right|_{F = F_{m-1}(\mathbf{x}_i)}, \quad i = 1,\dots,n. $$
   2. Обучить регрессионную модель $h_m$ (обычно дерево) на данных $\{(\mathbf{x}_i, r_{im})\}_{i=1}^n$.
   3. Найти шаг $\nu_m$ (line search или фиксированный $\nu$).
   4. Обновить модель:  
      $$ F_m(\mathbf{x}) = F_{m-1}(\mathbf{x}) + \nu_m \, h_m(\mathbf{x}). $$

3. **Выдать** $F_M(\mathbf{x})$ как итоговую модель.

#### Углублённый взгляд: Ньютоновский бустинг (второй порядок)

Сходимость можно улучшить, используя разложение второго порядка в точке $F_{m-1}(\mathbf{x}_i)$:

$$
L(y_i, F_{m-1} + f) \approx L(y_i, F_{m-1}) + g_i f + \frac{1}{2} h_i f^2,
$$

где

$$
g_i = \frac{\partial L(y_i, F)}{\partial F}\Big|_{F_{m-1}(\mathbf{x}_i)}, \qquad
h_i = \frac{\partial^2 L(y_i, F)}{\partial F^2}\Big|_{F_{m-1}(\mathbf{x}_i)}.
$$

При фиксированных $g_i$, $h_i$ квадратичная форма по $f$ достигает минимума при

$$
f = -\frac{g_i}{h_i}.
$$

Таким образом, в качестве целевых значений для листьев дерева можно брать величины $-g_i/h_i$, а само дерево строить с учётом весов $h_i$ (взвешенный метод наименьших квадратов). Именно этот подход применяется в XGBoost и LightGBM.

Выпишем $g_i$ и $h_i$ для основных функций потерь.

- **Квадратичная ошибка** $L = \tfrac{1}{2}(y - F)^2$:  
  $g_i = -(y_i - F_{m-1}(\mathbf{x}_i)), \quad h_i = 1$.  
  Оптимальное предсказание в листе: $f = y_i - F_{m-1}(\mathbf{x}_i)$ — средний остаток.

- **Log‑loss, $y \in \{-1, +1\}$** $L = \log(1 + \exp(-2yF))$:  
  Уже вычислено: $g_i = \frac{-2y_i}{1 + \exp(2y_i F_{m-1})}$. Вторая производная:  
  $$
  h_i = \frac{\partial g_i}{\partial F}
  = \frac{4 \exp(2y_i F_{m-1})}{\bigl(1 + \exp(2y_i F_{m-1})\bigr)^2}
  = 4 \, \sigma(2F_{m-1})\,\bigl(1 - \sigma(2F_{m-1})\bigr),
  $$
  где $\sigma(z) = 1/(1+e^{-z})$. Видно, что $h_i > 0$, что гарантирует выпуклость.

- **Логистическая регрессия, $y \in \{0,1\}$** $L = -[y\log\sigma(F) + (1-y)\log(1-\sigma(F))]$:  
  $g_i = \sigma(F_{m-1}) - y_i$,  
  $h_i = \sigma(F_{m-1})(1 - \sigma(F_{m-1}))$.

Ньютоновский шаг $-g_i/h_i$ для логистической регрессии даёт

$$
f = \frac{y_i - \sigma(F_{m-1})}{\sigma(F_{m-1})(1 - \sigma(F_{m-1}))},
$$

что в точности соответствует шагу метода Ньютона‑Рафсона для обычной логистической регрессии. Это позволяет бустингу быстрее достигать оптимума по сравнению с градиентным спуском первого порядка.

---

### Вопросы для самопроверки

#### Для начинающих

1. Что такое градиентный бустинг и в чём его принципиальное отличие от бэггинга и случайного леса?
2. Зачем нужен learning rate $\nu$ и как его выбор влияет на качество модели?
3. Что такое псевдо‑остатки? Как они вычисляются для квадратичной функции потерь и для логистической регрессии?
4. Почему градиентный бустинг склонен к переобучению и какие механизмы используют для его предотвращения?

#### Для профессионалов

1. Выведите псевдо‑остатки для log‑loss в обеих параметризациях ($y \in \{-1,+1\}$ и $y \in \{0,1\}$) и покажите их связь с остатками линейной регрессии.
2. Объясните, как shrinkage ($\nu < 1$) действует в качестве регуляризации. Сравните этот эффект с $L_2$-регуляризацией в линейных моделях.
3. Докажите, что градиентный бустинг является функциональным градиентным спуском в гильбертовом пространстве функций, порождённом ядром, задаваемым ансамблем деревьев.
4. Сравните сходимость градиентного бустинга первого порядка и Ньютоновского бустинга. В каких случаях предпочтителен каждый из них?

## Градиентный бустинг: реализация с нуля и сравнение с библиотеками

После теоретического обоснования градиентного бустинга как функционального градиентного спуска перейдём к его практической реализации. Мы напишем с нуля регрессионную и классификационную версии GBM, проверим их корректность на синтетических и реальных данных, исследуем влияние гиперпараметров и кратко познакомимся с основными промышленными библиотеками.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, accuracy_score, log_loss
```

### 1. Реализация Gradient Boosting Machine (GBM) с нуля

Идея GBM заключается в последовательном добавлении слабых моделей (обычно неглубоких деревьев), каждая из которых обучается на псевдо‑остатках, равных антиградиенту функции потерь по текущему предсказанию. Мы реализуем два класса: для регрессии (квадратичная ошибка) и для бинарной классификации (логистическая функция потерь).

#### Регрессия

```python
class GradientBoostingRegressor:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None

    def fit(self, X, y):
        self.F0 = np.mean(y)                    # оптимальная константа для MSE
        F = np.full(len(y), self.F0)            # текущее предсказание
        self.trees = []
        for _ in range(self.n_estimators):
            residuals = y - F                   # отрицательный градиент для MSE
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X)
        return self

    def predict(self, X):
        F = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return F
```

Здесь на каждом шаге вычисляются обычные остатки $y - F_{m-1}(\mathbf{x})$, и дерево учится их предсказывать. Добавление предсказаний дерева с малым шагом $\nu$ постепенно уменьшает ошибку.

#### Бинарная классификация (логистическая регрессия)

Для классификации используем сигмоидную функцию $\sigma(F) = 1/(1+e^{-F})$ и логарифмическую функцию потерь. Псевдо‑остатки равны $y_i - \sigma(F(\mathbf{x}_i))$.

```python
class GradientBoostingClassifier:
    def __init__(self, n_estimators=100, learning_rate=0.1, max_depth=3):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.trees = []
        self.F0 = None

    @staticmethod
    def _sigmoid(F):
        return 1.0 / (1.0 + np.exp(-F))

    def fit(self, X, y):
        # доля положительного класса
        p = np.mean(y)
        self.F0 = np.log(p / (1 - p))           # оптимальная константа для log loss
        F = np.full(len(y), self.F0)
        self.trees = []
        for _ in range(self.n_estimators):
            p_pred = self._sigmoid(F)
            residuals = y - p_pred              # отрицательный градиент
            tree = DecisionTreeRegressor(max_depth=self.max_depth)
            tree.fit(X, residuals)
            self.trees.append(tree)
            F += self.learning_rate * tree.predict(X)
        return self

    def predict_proba(self, X):
        F = np.full(X.shape[0], self.F0)
        for tree in self.trees:
            F += self.learning_rate * tree.predict(X)
        return self._sigmoid(F)

    def predict(self, X):
        proba = self.predict_proba(X)
        return (proba >= 0.5).astype(int)
```

Инициализация $F_0 = \log\frac{\bar{p}}{1-\bar{p}}$ соответствует решению задачи $\arg\min_\gamma \sum_i L(y_i, \gamma)$ для логистической функции потерь.

### 2. Проверка корректности

Сравним нашу реализацию с `GradientBoostingRegressor` и `GradientBoostingClassifier` из `sklearn`, установив одинаковые параметры и убедившись в близости результатов.

#### Регрессия

```python
# Синтетические данные
np.random.seed(0)
n = 200
x = np.linspace(-3, 3, n)
y = x**2 + np.sin(5 * x) + np.random.normal(0, 0.3, n)
X = x.reshape(-1, 1)

# Разделение
X_train, X_test = X[:150], X[150:]
y_train, y_test = y[:150], y[150:]

# Наша модель
our_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=3)
our_gbr.fit(X_train, y_train)
our_pred = our_gbr.predict(X_test)
our_mse = mean_squared_error(y_test, our_pred)

# sklearn
from sklearn.ensemble import GradientBoostingRegressor as SKGBR
sk_gbr = SKGBR(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
sk_gbr.fit(X_train, y_train)
sk_pred = sk_gbr.predict(X_test)
sk_mse = mean_squared_error(y_test, sk_pred)

print(f"Our GBR MSE: {our_mse:.4f}")
print(f"sklearn GBR MSE: {sk_mse:.4f}")
```

Визуализируем эволюцию предсказаний с ростом числа итераций:

```python
# Промежуточные предсказания для нескольких шагов
steps = [1, 5, 10, 50, 100]
plt.figure(figsize=(12, 6))
for i, step in enumerate(steps):
    model = GradientBoostingRegressor(n_estimators=step, learning_rate=0.1, max_depth=3)
    model.fit(X_train, y_train)
    y_plot = model.predict(x.reshape(-1, 1))
    plt.subplot(2, 3, i+1)
    plt.scatter(X_train, y_train, alpha=0.4)
    plt.plot(x, y_plot, 'r-')
    plt.title(f'n_estimators={step}')
plt.tight_layout()
plt.show()
```

Графики показывают, как простая константа постепенно превращается в сложную функцию, всё лучше приближая обучающие данные.

#### Классификация

```python
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier as SKGBC

data = load_breast_cancer()
X_clf, y_clf = data.data, data.target
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_clf, y_clf, test_size=0.3, random_state=42)

our_clf = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3)
our_clf.fit(Xc_train, yc_train)
our_acc = accuracy_score(yc_test, our_clf.predict(Xc_test))
our_ll = log_loss(yc_test, our_clf.predict_proba(Xc_test))

sk_clf = SKGBC(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=0)
sk_clf.fit(Xc_train, yc_train)
sk_acc = accuracy_score(yc_test, sk_clf.predict(Xc_test))
sk_ll = log_loss(yc_test, sk_clf.predict_proba(Xc_test))

print(f"Our GBC: accuracy={our_acc:.4f}, log_loss={our_ll:.4f}")
print(f"sklearn GBC: accuracy={sk_acc:.4f}, log_loss={sk_ll:.4f}")
```

Результаты будут близки, подтверждая корректность нашей реализации.

### 3. Анализ влияния learning rate и числа деревьев

Параметр `learning_rate` ($\nu$) контролирует вклад каждого дерева. При малых $\nu$ обучение замедляется, но, как правило, достигает лучшего обобщения при достаточно большом числе итераций. Проиллюстрируем это на регрессионном примере: зафиксируем `n_estimators=500` и будем варьировать $\nu$.

```python
learning_rates = [0.01, 0.05, 0.1, 0.5, 1.0]
train_errors = []
test_errors = []
for lr in learning_rates:
    model = GradientBoostingRegressor(n_estimators=500, learning_rate=lr, max_depth=3)
    model.fit(X_train, y_train)
    train_errors.append(mean_squared_error(y_train, model.predict(X_train)))
    test_errors.append(mean_squared_error(y_test, model.predict(X_test)))

plt.figure(figsize=(8,5))
plt.plot(learning_rates, train_errors, 'bo-', label='train')
plt.plot(learning_rates, test_errors, 'rs-', label='test')
plt.xscale('log')
plt.xlabel('learning_rate')
plt.ylabel('MSE')
plt.legend()
plt.title('Влияние learning_rate при фиксированном числе итераций (500)')
plt.show()
```

График показывает, что при слишком большом $\nu$ модель переобучается (низкая ошибка на обучении, высокая на тесте). Уменьшение $\nu$ действует как регуляризация: тестовая ошибка сначала снижается, но при чрезмерно малом $\nu$ (и фиксированном $M=500$) может возрасти, так как модель не успевает сойтись. В реальных задачах число деревьев подбирают совместно с $\nu$.

### 4. Введение в библиотеки: XGBoost, LightGBM, CatBoost (краткий обзор)

Промышленные реализации градиентного бустинга — XGBoost, LightGBM и CatBoost — существенно оптимизируют базовый алгоритм, добавляя:

- учёт вторых производных (Ньютоновский бустинг),
- встроенную $L_1/L_2$ регуляризацию,
- эффективную работу с пропущенными значениями,
- специальную обработку категориальных признаков,
- параллельное построение деревьев и гистограммный поиск расщеплений.

Детальный разбор этих усовершенствований будет дан в следующих разделах. Здесь приведём лишь пример вызова для оценки скорости и качества на примере California Housing.

```python
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
import time

housing = fetch_california_housing()
Xh, yh = housing.data, housing.target
Xh_train, Xh_test, yh_train, yh_test = train_test_split(
    Xh, yh, test_size=0.2, random_state=42)

def benchmark_model(name, model, Xt, yt, Xte, yte):
    start = time.time()
    model.fit(Xt, yt)
    train_time = time.time() - start
    pred = model.predict(Xte)
    mse = mean_squared_error(yte, pred)
    print(f"{name}: MSE={mse:.4f}, время обучения={train_time:.2f} сек")
    return mse, train_time

# Наша простая реализация
our_gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, max_depth=4)
benchmark_model("Our GBM", our_gbr, Xh_train, yh_train, Xh_test, yh_test)

# XGBoost
try:
    from xgboost import XGBRegressor
    xgb = XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbosity=0)
    benchmark_model("XGBoost", xgb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("XGBoost не установлен")

# LightGBM
try:
    from lightgbm import LGBMRegressor
    lgb = LGBMRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbose=-1)
    benchmark_model("LightGBM", lgb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("LightGBM не установлен")

# CatBoost
try:
    from catboost import CatBoostRegressor
    ctb = CatBoostRegressor(n_estimators=100, learning_rate=0.1, max_depth=4, verbose=0)
    benchmark_model("CatBoost", ctb, Xh_train, yh_train, Xh_test, yh_test)
except ImportError:
    print("CatBoost не установлен")
```

Скорость работы библиотек на порядки превосходит нашу наивную реализацию благодаря упомянутым выше алгоритмическим и инженерным оптимизациям.

---

### Вопросы для самопроверки

#### Для начинающих
1. Почему начальное предсказание $F_0$ в регрессионном GBM устанавливается равным среднему значению $y$?
2. Как в классификационном GBM вычисляются псевдо‑остатки и почему они отличаются от обычных остатков?
3. Каким образом малый learning rate помогает предотвратить переобучение? Приходится ли за это платить?

#### Для профессионалов
1. Объясните, почему в бинарной классификации с логарифмической функцией потерь начальное приближение берётся равным $\log\frac{\bar{p}}{1-\bar{p}}$. Выведите это значение из условия минимизации эмпирического риска.
2. Сравните вычислительную сложность нашей реализации и промышленных библиотек. За счёт каких механизмов XGBoost/LightGBM достигают высокой скорости?
3. Как можно ускорить вычисление псевдо‑остатков в нашей реализации, не меняя алгоритмической сути (подсказка: векторизация, предварительное выделение памяти)?

## Современный градиентный бустинг: XGBoost, LightGBM, CatBoost — алгоритмические инновации и математика

Развитие идей градиентного бустинга привело к появлению высокопроизводительных библиотек, которые не просто реализуют базовый алгоритм, но и вводят принципиальные усовершенствования: регуляризацию, использование вторых производных, специальные приёмы для работы с большими данными и категориальными признаками. В этом разделе мы разберём математический фундамент трёх наиболее влиятельных реализаций — XGBoost, LightGBM и CatBoost — и проведём их сравнительный анализ на реальной задаче.

### 1. XGBoost: регуляризация и Newton boosting

XGBoost (eXtreme Gradient Boosting), предложенный Ченом и Гестриным (Chen & Guestrin, 2016), строит аддитивную модель

$$
\hat{y}_i = \sum_{t=1}^M f_t(\mathbf{x}_i),\qquad f_t \in \mathcal{F},
$$

где $\mathcal{F}$ — пространство решающих деревьев. Главное нововведение — явная регуляризация сложности каждого дерева и использование вторых производных функции потерь (Newton boosting).

#### Целевая функция с регуляризацией

Полная целевая функция, минимизируемая на шаге $t$, имеет вид

$$
\text{Obj}^{(t)} = \sum_{i=1}^n L\bigl(y_i, \hat{y}_i^{(t-1)} + f_t(\mathbf{x}_i)\bigr) + \sum_{k=1}^t \Omega(f_k),
$$

где $\Omega(f_k)$ — штраф за сложность $k$-го дерева. Для одного дерева $f$ с $T$ листьями и вектором весов листьев $\mathbf{w} \in \mathbb{R}^T$ полагают

$$
\Omega(f) = \gamma T + \frac{1}{2}\lambda \|\mathbf{w}\|^2.
$$

Параметр $\gamma$ штрафует число листьев, а $\lambda$ — $L_2$-регуляризацию весов (обычно на практике добавляют и $L_1$-регуляризацию $\alpha \|\mathbf{w}\|_1$, но для простоты мы ограничимся $L_2$).

#### Ньютоновское разложение и оптимальные веса листьев

Разложим функцию потерь в точке $\hat{y}_i^{(t-1)}$ до второго порядка:

$$
L(y_i, \hat{y}_i^{(t-1)} + f_t(\mathbf{x}_i)) \approx L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t(\mathbf{x}_i)^2,
$$

где

$$
g_i = \left.\frac{\partial L(y_i, \hat{y})}{\partial \hat{y}}\right|_{\hat{y} = \hat{y}_i^{(t-1)}},\qquad
h_i = \left.\frac{\partial^2 L(y_i, \hat{y})}{\partial \hat{y}^2}\right|_{\hat{y} = \hat{y}_i^{(t-1)}}.
$$

Суммируя по всем объектам и добавляя регуляризацию, получаем приближённую целевую функцию для дерева $f_t$:

$$
\tilde{\text{Obj}}^{(t)} = \sum_{i=1}^n \left[g_i f_t(\mathbf{x}_i) + \frac{1}{2} h_i f_t(\mathbf{x}_i)^2\right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2.
$$

Пусть дерево $f_t$ отображает каждый объект в один из $T$ листьев. Обозначим $I_j = \{i \mid \mathbf{x}_i \text{ попадает в лист } j\}$. Тогда $f_t(\mathbf{x}_i) = w_j$ для $i \in I_j$, и мы можем переписать сумму по листьям:

$$
\tilde{\text{Obj}}^{(t)} = \sum_{j=1}^T \left[ w_j \sum_{i \in I_j} g_i + \frac{1}{2} w_j^2 \sum_{i \in I_j} h_i \right] + \gamma T + \frac{1}{2}\lambda \sum_{j=1}^T w_j^2.
$$

При фиксированной структуре дерева (известны множества $I_j$) можно аналитически найти оптимальный вес каждого листа, минимизирующий квадратичную форму. Приравнивая производную по $w_j$ к нулю:

$$
\frac{\partial}{\partial w_j} = \sum_{i \in I_j} g_i + w_j \sum_{i \in I_j} h_i + \lambda w_j = 0 \;\Longrightarrow\;
w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}.
$$

Подставляя $w_j^*$ обратно, получаем **оптимальное значение целевой функции** для данной структуры:

$$
\tilde{\text{Obj}}^* = -\frac{1}{2} \sum_{j=1}^T \frac{(\sum_{i \in I_j} g_i)^2}{\sum_{i \in I_j} h_i + \lambda} + \gamma T.
$$

Это выражение служит критерием качества разбиения — аналогом impurity в классическом CART. При поиске расщепления оценивается уменьшение $\tilde{\text{Obj}}^*$, называемое **gain**:

$$
\text{Gain} = \frac{1}{2}\left[ \frac{(\sum_{i \in I_L} g_i)^2}{\sum_{i \in I_L} h_i + \lambda} + \frac{(\sum_{i \in I_R} g_i)^2}{\sum_{i \in I_R} h_i + \lambda} - \frac{(\sum_{i \in I} g_i)^2}{\sum_{i \in I} h_i + \lambda} \right] - \gamma.
$$

Расщепление выполняется только если Gain > 0, что автоматически включает в себя pruning.

#### Вычислительные оптимизации

XGBoost использует специальную структуру данных (block structure) для хранения отсортированных значений каждого признака, позволяя быстро находить кандидаты на расщепление. Также реализовано эффективное обучение с пропущенными данными: для каждого узла выбирается направление «по умолчанию» (в левое или правое поддерево) на основе того, какое даёт меньший лосс. Поддержка разреженных форматов (CSR) и параллелизм на уровне построения деревьев делают XGBoost одной из самых масштабируемых реализаций.

### 2. LightGBM: GOSS и EFB

LightGBM (Light Gradient Boosting Machine), предложенный Ке и соавторами (Ke et al., 2017), решает проблему масштабирования градиентного бустинга на очень большие обучающие выборки и высокоразмерные пространства признаков. Два ключевых нововведения — Gradient‑based One‑Side Sampling (GOSS) и Exclusive Feature Bundling (EFB).

#### Gradient‑based One‑Side Sampling (GOSS)

Традиционный бустинг использует все объекты для вычисления градиентов, что дорого. Идея GOSS: объекты с большими градиентами (те, которые сильнее всего «ошибаются» и, следовательно, вносят основной вклад в обучение) должны сохраняться полностью, а из объектов с малыми градиентами можно взять случайную подвыборку, умножив их градиенты и гессианы на коэффициент для сохранения несмещённости оценки распределения данных.

Формально, пусть отсортированы модули градиентов: $|g_{(1)}| \le |g_{(2)}| \le \dots \le |g_{(n)}|$. Зададим долю $a$ (например, 0.2) для объектов с наибольшими градиентами и долю $b$ для случайной подвыборки из остальных. Тогда:

- Сохраняем все объекты с наибольшими градиентами: множество $A$ размера $a n$.
- Из оставшихся $(1-a)n$ объектов случайно выбираем подмножество $B$ размера $b n$ (где $b$ — доля от общего числа объектов).
- При вычислении сумм градиентов и гессианов для объектов из $B$ их величины умножаются на $\frac{1-a}{b}$.

Этот приём позволяет резко сократить объём данных для построения одного дерева, при этом оценка градиента остаётся асимптотически несмещённой, что доказано в оригинальной статье через анализ влияния на информацию Фишера.

#### Exclusive Feature Bundling (EFB)

В разреженных данных (например, после one‑hot кодирования) многие признаки являются взаимоисключающими (exclusive), то есть принимают ненулевые значения одновременно лишь для небольшого числа объектов. LightGBM объединяет такие признаки в один «пучок» (bundle), что уменьшает размерность без потери информации. Задача сводится к графовой раскраске: признаки — вершины, рёбра между невзаимоисключающими признаками. Для эффективного приближённого решения используется жадный алгоритм. После объединения каждый исходный признак кодируется смещёнными значениями в общем диапазоне, гарантируя различимость.

#### Leaf‑wise growth (рост по листьям)

В отличие от XGBoost и классического GBM, которые растят дерево по уровням (level‑wise), LightGBM выбирает лист с максимальным уменьшением потерь и расщепляет именно его (leaf‑wise). Такой подход порождает более глубокие и асимметричные деревья, которые при одинаковом числе листьев часто дают меньшую ошибку, но могут привести к переобучению на малых данных. Поэтому в LightGBM обязательно задаётся ограничение на максимальную глубину.

### 3. CatBoost: ordered boosting и обработка категориальных признаков

CatBoost (Categorical Boosting), разработанный Прохоровым и соавторами (Prokhorenkova et al., 2018), фокусируется на двух проблемах: надёжная обработка категориальных признаков и устранение систематического смещения градиентов (target leakage).

#### Проблема target leakage и ordered boosting

В стандартном градиентном бустинге при вычислении псевдо‑остатков и, особенно, при преобразовании категориальных признаков в числовые (target statistics) используется информация о целевой переменной текущего объекта, что ведёт к смещению оценок и переобучению (target leakage). CatBoost решает эту проблему с помощью **ordered boosting** — модифицированного алгоритма, который имитирует обучение на независимых данных.

Основная идея: генерируется случайная перестановка обучающих объектов. Для каждого объекта $i$ модель, используемая для вычисления псевдо‑остатков, обучается только на объектах, предшествующих $i$ в этой перестановке. Таким образом, для объекта $i$ получаем честный прогноз, не содержащий информации о $y_i$. На практике, для вычислительной эффективности, используется несколько перестановок и специальная схема «обучения без i-го объекта» (permutation‑based gradient boosting).

#### Обработка категориальных признаков: target statistics со сглаживанием

Для категориального признака CatBoost вычисляет **target statistics** — среднее значение целевой переменной по каждой категории, но с добавлением априорного сглаживания для борьбы с шумом при редких категориях. Формула для числового представления категории $k$:

$$
\hat{x}_k = \frac{\sum_{j=1}^n [x_j = k] \cdot y_j + a \cdot p_{\text{global}}}{n_k + a},
$$

где $n_k$ — число объектов с категорией $k$, $p_{\text{global}}$ — глобальное среднее целевой переменной (например, среднее $y$ для регрессии или базовая вероятность для классификации), $a > 0$ — параметр сглаживания, интерпретируемый как «виртуальные наблюдения» с известным средним. Эта формула выводится из байесовской оценки с априорным распределением, и аналогична сглаживанию Лапласа.

Важно, что эта статистика вычисляется с учётом порядка, то есть для объекта $i$ используются только предшествующие объекты в перестановке, что исключает утечку целевой переменной.

#### Симметричные деревья

CatBoost использует **симметричные деревья** (balanced trees): на каждом уровне все узлы используют один и тот же признак и порог для расщепления. Это упрощает и ускоряет как обучение, так и инференс (предсказание можно представить в виде быстрой табличной функции). Хотя симметричные деревья менее гибки, чем асимметричные, на практике они показывают конкурентоспособное качество, особенно в сочетании с ordered boosting.

### 4. Сравнительный эксперимент

Проведём сравнение трёх библиотек на наборе данных California Housing (задача регрессии). Для чистоты эксперимента фиксируем число итераций (1000), learning rate (0.05), максимальную глубину деревьев (6). Оценим время обучения и качество (RMSE).

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import time

# Импортируем библиотеки
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor

# Загрузка данных
data = fetch_california_housing()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

models = {
    'XGBoost': XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                            verbosity=0, random_state=42),
    'LightGBM': LGBMRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                              verbose=-1, random_state=42),
    'CatBoost': CatBoostRegressor(n_estimators=1000, learning_rate=0.05, max_depth=6,
                                  verbose=0, random_seed=42)
}

results = {}
for name, model in models.items():
    start = time.time()
    model.fit(X_train, y_train)
    elapsed = time.time() - start
    pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, pred))
    results[name] = {'RMSE': rmse, 'Time (s)': elapsed}
    print(f"{name:10s} | RMSE: {rmse:.4f} | Time: {elapsed:.2f} s")
```

Визуализируем результаты столбчатыми диаграммами.

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(results.keys())
rmse_vals = [results[n]['RMSE'] for n in names]
time_vals = [results[n]['Time (s)'] for n in names]

axes[0].bar(names, rmse_vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[0].set_title('RMSE на тесте')
axes[0].set_ylabel('RMSE')

axes[1].bar(names, time_vals, color=['#1f77b4', '#ff7f0e', '#2ca02c'])
axes[1].set_title('Время обучения')
axes[1].set_ylabel('Секунды')

plt.tight_layout()
plt.show()
```

Обычно LightGBM оказывается самым быстрым на больших наборах данных, CatBoost немного медленнее, но может давать лучший результат при наличии категориальных признаков, а XGBoost демонстрирует стабильно высокое качество за счёт сильной регуляризации. В данном эксперименте категориальных признаков нет, поэтому различия в качестве невелики, а время обучения отражает различия в реализации.






### 5. Сравнительная таблица: XGBoost vs LightGBM vs CatBoost

Все три библиотеки постоянно развиваются, и на 2026 год каждая из них предлагает уникальные оптимизации и функции. Таблица ниже систематизирует их ключевые различия.

| Характеристика | XGBoost | LightGBM | CatBoost |
| :--- | :--- | :--- | :--- |
| **Актуальная версия** | 3.2.0 (февраль 2026) | 4.6.0 (релиз; также встречается упоминание версии 4.6.0.99) | 1.2.10 |
| **Способ роста дерева** | **Level-wise** (глубинная). Растет уровень за уровнем, обеспечивая сбалансированность. | **Leaf-wise** (по листьям). Выбирает для расширения лист с наибольшим *gain*, что ведет к более глубоким и несимметричным деревьям, потенциально повышая точность, но требуя контроля над `num_leaves` во избежание переобучения. | **Симметричные** (Oblivious). На каждом уровне дерева используется один и тот же признак для расщепления, что значительно ускоряет инференс и сокращает время предсказания. |
| **Регуляризация** | Сильная и гибкая: $L_1$ (`alpha`), $L_2$ (`lambda`) на веса листьев, штраф на число листьев (`gamma`). | Включает $L_2$ регуляризацию (`reg_lambda`), $L_1` (`reg_alpha`), а также параметры, контролирующие сложность дерева (`min_gain_to_split`, `min_child_samples`). | $L_2$ регуляризация (`l2_leaf_reg`). Склонность к переобучению сдерживается архитектурой симметричных деревьев. |
| **Категориальные признаки** | **Поддерживаются нативно с версии 3.1.0** (2025). Использует механизм *categorical re-coder*, автоматически сохраняя кодировку категорий в обученной модели. | **Поддерживаются нативно**. Не требует one-hot кодирования. Для поиска оптимального разбиения используется специализированный алгоритм на основе гистограмм, особенно эффективный с высококардинальными признаками. | **Ключевое преимущество**. Исключительно сильная поддержка. Использует передовой метод *Ordered Target Statistics*, который эффективно предотвращает утечку целевой переменной (target leakage) во время кодирования. |
| **Пропущенные значения** | **Обрабатываются автоматически** на этапе построения дерева («sparsity-aware split finding»). Алгоритм сам определяет оптимальное направление для ветви с пропусками. | **Обрабатываются автоматически**, что является одной из ключевых особенностей алгоритма. | **Обрабатываются автоматически**. |
| **Подвыборка объектов** | `subsample` — случайная выборка объектов без возврата перед построением каждого дерева. | `bagging_fraction` — случайная выборка объектов. Также доступен продвинутый метод **GOSS (Gradient-based One-Side Sampling)**, который оставляет объекты с большими градиентами и случайно подвыбирает объекты с малыми, сохраняя оценку градиента практически несмещённой. | `subsample` — случайная выборка объектов. |
| **Подвыборка признаков** | Высокая гибкость: `colsample_bytree`, `colsample_bylevel`, `colsample_bynode` для различных уровней детализации. | `feature_fraction` — случайный выбор доли признаков для каждого дерева. | `rsm` (random subspace method) — реализация метода случайных подпространств. |
| **Ускорение** | **Предварительная сортировка (Block structure)**, эффективная **GPU поддержка** (`gpu_hist`), параллельные вычисления. | **Гистограммный алгоритм**, **GOSS**, техника объединения разреженных признаков (**EFB**), что делает его одним из самых быстрых на больших данных. Поддержка GPU. | **Ordered Boosting** для устранения смещения, **симметричные деревья** для быстрых предсказаний. Эффективная поддержка GPU. Включает поддержку текстовых и эмбеддинг-признаков. |
| **Поддержка GPU** | **Да**. Требует установки из исходников или специальной сборки. | **Да**. Требует установки из исходников или специальной сборки (`lightgbm --gpu`). Включает поддержку NVIDIA Blackwell через CUDA. | **Да**. Обеспечивается "из коробки" в официальных пакетах (`pip install catboost`). В версии 1.2.10 есть поддержка устройств с CUDA compute capability 3.5 и новее. |
| **Ключевая инновация версии** (по состоянию на начало 2026 г.) | **Версия 3.2.0** (февраль 2026) представила **векторные листья (Vector Leaf)** для нативного multi-output обучения и оптимизации для GPU external memory. | **Версия 4.6.0** сфокусирована на GPU-ускорении, включая поддержку архитектуры NVIDIA Blackwell, и улучшении распределенного обучения. | **Версия 1.2.10** делает упор на стабильность, поддержку новых типов признаков (текстовые, эмбеддинги) и расширенную аналитику (Object Importance). |
| **Типичный сценарий** | **Универсальный солдат**: для большинства задач классификации/регрессии, особенно когда нужна тонкая регуляризация (L1/L2) и интерпретируемость через SHAP. | **Спринтер**: для очень больших наборов данных (миллионы строк), где скорость обучения критична. Требует осторожной настройки `num_leaves`. | **Специалист по гетерогенным данным**: для данных с большим количеством категориальных признаков. Лучший выбор, когда важна стабильность и качество "из коробки". |

### Вопросы для самопроверки

#### Для начинающих
1. Назовите три самые популярные библиотеки градиентного бустинга и кратко охарактеризуйте их.
2. В чём отличие LightGBM от XGBoost по способу построения дерева (leaf‑wise vs level‑wise)?
3. Почему CatBoost хорошо работает с категориальными признаками без предварительного one‑hot кодирования?
4. Что такое early stopping и зачем он используется при обучении бустинга?

#### Для профессионалов
1. Выведите оптимальные веса листьев и значение целевой функции для XGBoost (формулы для $w_j^*$ и $\tilde{\text{Obj}}^*$). Покажите, как получается критерий Gain для расщепления.
2. Объясните, как GOSS корректирует смещение оценки градиента при подвыборке объектов с малыми градиентами. Зачем нужно умножать на коэффициент $(1-a)/b$?
3. Опишите алгоритм EFB и условия, при которых признаки могут быть объединены в один пучок без потери информации. Как задача упаковки признаков сводится к раскраске графа?
4. Сравните смещение и дисперсию моделей, получаемых при leaf‑wise и level‑wise росте деревьев. Почему leaf‑wise требует дополнительной регуляризации?

## Интерпретация ансамблей: SHAP, PDP, ICE и заключение главы

Ансамблевые методы, при всех их предсказательных достоинствах, часто воспринимаются как «чёрные ящики»: большое число деревьев делает модель непрозрачной. Однако современный инструментарий позволяет не только оценить глобальную важность признаков (раздел 9.3), но и ответить на вопрос, *почему* модель выдала конкретное предсказание и как признаки влияют на выход ансамбля в среднем. В этом заключительном разделе мы рассмотрим Partial Dependence Plots (PDP), Individual Conditional Expectation (ICE) и Shapley Additive Explanations (SHAP), а затем подведём итог всей главы, сравнив изученные методы и обозначив дальнейшие направления.

```python
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import PartialDependenceDisplay
import shap
```

### 1. Необходимость интерпретации и локальные объяснения

Важность признаков (MDI, permutation importance) даёт глобальную картину, но не объясняет отдельное предсказание и не показывает форму зависимости. В задачах, где требуется прозрачность решений (медицина, кредитный скоринг), нужны методы, вскрывающие внутреннюю логику ансамбля. Partial Dependence Plots описывают среднее маржинальное влияние признака, а SHAP-значения справедливо распределяют вклад каждого признака в конкретный прогноз, опираясь на кооперативную теорию игр.

### 2. Partial Dependence Plots (PDP) и Individual Conditional Expectation (ICE)

Пусть $S \subset \{1,\dots,D\}$ — подмножество признаков, а $\overline{S}$ — дополнение. Модель $\hat{f}$ обучена на $\{(\mathbf{x}_i, y_i)\}_{i=1}^n$. Частичная зависимость для $S$ определяется как

$$
\hat{f}_S(\mathbf{x}_S) = \frac{1}{n} \sum_{i=1}^{n} \hat{f}\bigl(\mathbf{x}_S,\; \mathbf{x}_{\overline{S}}^{(i)}\bigr),
$$

где $\mathbf{x}_{\overline{S}}^{(i)}$ — значения остальных признаков из $i$-го обучающего наблюдения. На практике берут подвыборку, так как усреднение по всем $n$ объектам требует $n \cdot |\text{grid}|$ предсказаний. PDP показывает маржинальное влияние признаков $S$, усреднённое по распределению остальных признаков.

**ICE-кривые** (Individual Conditional Expectation) — это графики $\hat{f}(\mathbf{x}_S, \mathbf{x}_{\overline{S}}^{(i)})$ для каждого $i$ (или для случайной подвыборки). Они позволяют увидеть гетерогенность: различное влияние признака на разные объекты. Линия PDP получается усреднением ICE-кривых.

**Ограничения.** При сильной корреляции признаков усреднение может давать нереалистичные комбинации (например, рост и вес, противоречащие друг другу). В таких случаях полезна альтернатива — Accumulated Local Effects (ALE), которая усредняет локальные приращения предсказания, но в рамках данной главы мы ограничимся PDP/ICE.

**Пример.** Построим PDP и ICE для случайного леса на breast_cancer.

```python
data = load_breast_cancer()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

rf = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=42)
rf.fit(X_train, y_train)

features = [0, 1]  # mean radius, mean texture
fig, ax = plt.subplots(figsize=(8,5))
PartialDependenceDisplay.from_estimator(rf, X_train, features, ax=ax, kind='both', subsample=100)
plt.suptitle('PDP и ICE для двух признаков')
plt.show()
```

График показывает, как изменяется средняя предсказанная вероятность класса с ростом признака, и демонстрирует разброс индивидуальных кривых.

### 3. SHAP (SHapley Additive exPlanations)

SHAP (Lundberg & Lee, 2017) основан на значениях Шепли из теории кооперативных игр. Для одного наблюдения с вектором признаков $\mathbf{x}$ каждый признак рассматривается как игрок, а модель $\hat{f}$ — как характеристическая функция. Цель — справедливо распределить разницу $\hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(X)]$ между признаками.

#### Определение Shapley value

Пусть $M$ — множество всех признаков ($|M| = D$). Для подмножества $S \subseteq M$ определим функцию ценности

$$
v(S) = \mathbb{E}\bigl[\hat{f}(X) \mid X_S = \mathbf{x}_S\bigr],
$$

то есть среднее предсказание модели при фиксированных значениях признаков из $S$ равными $\mathbf{x}_S$, а остальные признаки маргинализуются по распределению обучающей выборки. Тогда Shapley value для признака $j$:

$$
\phi_j(v) = \sum_{S \subseteq M \setminus \{j\}} \frac{|S|!\, (D - |S| - 1)!}{D!} \bigl[ v(S \cup \{j\}) - v(S) \bigr].
\tag{1}
$$

Интуитивно: $\phi_j$ — средневзвешенный маржинальный вклад признака $j$ при всех возможных порядках добавления признаков.

Свойства Шепли:
- **Эффективность:** $\sum_{j=1}^D \phi_j = \hat{f}(\mathbf{x}) - \mathbb{E}[\hat{f}(X)]$.
- **Симметричность:** если $v(S \cup \{j\}) = v(S \cup \{k\})$ для всех $S$, то $\phi_j = \phi_k$.
- **Линейность:** для двух игр $v_1, v_2$ значение для $v_1+v_2$ равно сумме значений.
- **Фиктивный игрок:** если $v(S \cup \{j\}) = v(S)$ для всех $S$, то $\phi_j = 0$.

В контексте ML свойство эффективности особенно важно: сумма всех SHAP-значений для одного наблюдения в точности равна отклонению предсказания от среднего (базового значения), что даёт аддитивное объяснение.

#### Вычислительная сложность и TreeSHAP

Прямое вычисление (1) требует $2^D$ оценок, что невозможно для реальных задач. Для моделей на основе деревьев (в том числе ансамблей) разработан алгоритм **TreeSHAP**, вычисляющий точные Shapley values за полиномиальное время $O(T L D^2)$, где $T$ — число деревьев, $L$ — число листьев в дереве. Алгоритм рекурсивно обходит дерево, отслеживая число обучающих объектов, попадающих в каждый узел, и накапливает вклады, используя свойства покрытия.

> **Углублённый взгляд: идея TreeSHAP**
> В одном решающем дереве предсказание для $\mathbf{x}$ можно записать как сумму по листьям: $\hat{f}(\mathbf{x}) = \sum_{\ell} w_\ell \cdot \mathbf{1}_{\{\mathbf{x} \in R_\ell\}}$, где $w_\ell$ — значение в листе. Shapley value для признака $j$ выражается через взвешенную сумму по всем путям в дереве, где вес зависит от доли обучающих объектов, проходящих через каждое расщепление. Алгоритм вычисляет $\phi_j$ за один проход дерева, обновляя текущий вклад при каждом расщеплении по признаку $j$. Для ансамбля SHAP-значения усредняются по деревьям. Детали и доказательство корректности приведены в работе Lundberg et al. (2019).

### 4. Практическая визуализация SHAP

Используем библиотеку `shap` для анализа того же случайного леса на breast_cancer.

```python
explainer = shap.TreeExplainer(rf, X_train)
shap_values = explainer.shap_values(X_test)

# Summary plot
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names, show=False)
plt.title('SHAP summary plot')
plt.tight_layout()
plt.show()
```

Summary plot отображает для каждого признака распределение SHAP-значений всех тестовых объектов, с цветовой кодировкой величины признака. Видно, какие признаки наиболее важны (по оси — средний абсолютный SHAP), а также направление влияния: высокие значения признака (красные точки) сдвигают предсказание в сторону класса 1 или 0.

Bar plot средней важности:

```python
shap.summary_plot(shap_values[1], X_test, feature_names=data.feature_names, plot_type='bar', show=False)
plt.title('SHAP feature importance')
plt.tight_layout()
plt.show()
```

Waterfall plot для одного наблюдения (например, первого в тестовой выборке):

```python
shap.plots.waterfall(shap.Explanation(values=shap_values[1][0],
                     base_values=explainer.expected_value[1],
                     data=X_test[0], feature_names=data.feature_names))
```

Он показывает, как каждый признак отталкивает предсказание от базового значения (среднего) к итоговой вероятности.

Сравните SHAP-важности с MDI и permutation importance: SHAP обладает преимуществом согласованности (consistent), что делает его особенно надёжным.

### 5. Итоговое сравнение ансамблевых методов

Сведём характеристики основных методов в таблицу.

| Метод | Смещение | Дисперсия | Вычислительная сложность | Число гиперпараметров | Интерпретируемость |
|-------|----------|-----------|---------------------------|------------------------|---------------------|
| Одиночное дерево | Низкое | Высокая | $O(D \, n \log n)$ | Мало | Высокая (визуализация правил) |
| Бэггинг | Низкое | Снижена ($\rho\sigma^2$) | $O(B \cdot$ сложность дерева$)$ | Средне | Средняя (важность признаков) |
| Случайный лес | Низкое | Ещё ниже (декорреляция) | Как бэггинг, но с $m<D$ | Средне | Средняя (MDI + OOB) |
| Градиентный бустинг | Низкое (регулируется) | Низкая | $O(M \cdot$ сложность дерева$)$ | Много (learning rate, число итераций, глубина, регуляризация) | Низкая (SHAP восстанавливает) |

**Практические рекомендации:**
- Если нужен быстрый и неплохой baseline на табличных данных — случайный лес.
- Если требуется максимальная точность и есть ресурсы на настройку — градиентный бустинг (XGBoost/LightGBM/CatBoost).
- При большом количестве категориальных признаков особенно удобен CatBoost.
- Для сверхбольших выборок LightGBM предпочтителен по скорости.

### 6. Заключение и мост к глубокому обучению

Решающие деревья и их ансамбли остаются основным инструментом для табличных данных, где признаки имеют явный семантический смысл. Они не требуют масштабирования признаков, устойчивы к выбросам и способны улавливать нелинейные взаимодействия. Однако изображения, звук и текст содержат сырые данные высокой размерности, где решающие деревья неэффективны — там доминируют нейронные сети, автоматически извлекающие иерархические представления.

Современные исследования (TabNet, NODE) пытаются объединить интерпретируемость деревьев и мощь глубокого обучения, создавая дифференцируемые деревья и нейро-ансамбли. Таким образом, тема ансамблей не изолирована, а естественно вливается в более широкую картину машинного обучения.

---

### Вопросы для самопроверки (по всей главе)

#### Для начинающих
1. Чем решающее дерево принципиально отличается от линейной модели с точки зрения формы предсказаний?
2. Зачем случайный лес случайно выбирает признаки при каждом расщеплении?
3. В чём основная идея градиентного бустинга?
4. Назовите три популярные библиотеки градиентного бустинга.
5. Что такое SHAP-значения и какую информацию они дают о конкретном предсказании?
6. В какой ситуации вы выберете случайный лес, а в какой — градиентный бустинг?

#### Для профессионалов
1. Выведите формулу дисперсии ансамбля $\mathrm{Var}(\bar{f}) = \rho\sigma^2 + \frac{1-\rho}{B}\sigma^2$ и объясните роль корреляции $\rho$.
2. Опишите, как алгоритм TreeSHAP вычисляет Shapley values за полиномиальное время, используя структуру дерева.
3. Сравните Gradient-based One-Side Sampling (GOSS) и обычную случайную подвыборку объектов: как GOSS сохраняет несмещённость оценки градиента?
4. Объясните механизм смещения предсказаний при вычислении target statistics в бустинге и как ordered boosting в CatBoost решает эту проблему.
5. Докажите, что SHAP-значения удовлетворяют аксиомам Шепли (эффективность, симметричность, линейность, фиктивный игрок).

---

### Задания повышенной сложности

1. **Собственная OOB-ошибка.** Реализуйте вычисление out-of-bag ошибки для случайного леса с нуля (без использования `oob_score_`). Сравните её с оценкой, полученной кросс-валидацией.
2. **Вывод оптимальных весов XGBoost.** Для заданной структуры дерева и целевой функции с регуляризацией $L_2$ выведите аналитически оптимальные веса листьев и значение критерия. Убедитесь, что они совпадают с формулами (14)–(15).
3. **SHAP-взаимодействия.** С помощью `shap.dependence_plot` постройте график зависимости SHAP-значения для одного признака, раскрашенный значениями другого признака. Интерпретируйте обнаруженные взаимодействия.
4. **Устойчивость к шумовым признакам.** Добавьте к данным 30% случайных признаков и сравните точность случайного леса и градиентного бустинга. Проанализируйте, какой из методов лучше отсеивает неинформативные признаки.
5. **Упрощённый TreeSHAP.** Для одного решающего дерева (не ансамбля) реализуйте вычисление Shapley values с нуля по рекурсивному алгоритму TreeSHAP. Проверьте на синтетическом примере, что сумма SHAP-значений равна разности предсказания и среднего.